# Fusion 360 Dataset Exploration

Explore public CAD datasets and generate synthetic sketch training data for SketchForge.

**Datasets:**
- [Fusion 360 Gallery](https://github.com/AutodeskAILab/Fusion360GalleryDataset)
- DeepCAD / ABC Dataset

**Goals:**
1. Inspect CAD command sequence structure
2. Render edge maps from 3D models
3. Build `(sketch_image, CADSpec)` training pairs

In [ ]:
from pathlib import Path
import json
import numpy as np
import cv2
from PIL import Image, ImageFilter

DATA_DIR = Path('../data')
SKETCH_DIR = DATA_DIR / 'synthetic_sketches'
SKETCH_DIR.mkdir(parents=True, exist_ok=True)

print('Data directory:', DATA_DIR.resolve())

In [ ]:
def render_pencil_sketch(shape_image: np.ndarray) -> np.ndarray:
    """Convert a rendered shape image into a pencil-style sketch."""
    gray = cv2.cvtColor(shape_image, cv2.COLOR_BGR2GRAY) if shape_image.ndim == 3 else shape_image
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blur, 40, 120)
    inverted = 255 - edges
    paper = np.full_like(inverted, 245)
    noise = np.random.normal(0, 4, paper.shape).astype(np.int16)
    sketch = np.clip(paper - inverted + noise, 200, 255).astype(np.uint8)
    return sketch


def example_cad_spec() -> dict:
    return {
        'version': 1,
        'units': 'mm',
        'template': 'box',
        'sketches': [{'id': 'base', 'plane': 'XY', 'profile': [[0,0],[100,0],[100,50],[0,50]]}],
        'operations': [{'op': 'extrude', 'sketch': 'base', 'distance': 30}],
        'parameters': {'width': 100, 'depth': 50, 'height': 30, 'fillet_radius': 2},
    }

spec = example_cad_spec()
print(json.dumps(spec, indent=2))

In [ ]:
# Placeholder: load Fusion 360 Gallery JSON sequences when dataset is downloaded
# git clone https://github.com/AutodeskAILab/Fusion360GalleryDataset

F360_DIR = DATA_DIR / 'fusion360'
if F360_DIR.exists():
    sample_files = list(F360_DIR.rglob('*.json'))[:5]
    print(f'Found {len(sample_files)} sample JSON files')
    for path in sample_files:
        print(' -', path.name)
else:
    print('Download Fusion 360 Gallery into ml/data/fusion360 to explore sequences.')
    print('See: https://github.com/AutodeskAILab/Fusion360GalleryDataset')

In [ ]:
# Synthetic demo: create a simple box outline sketch and save training pair metadata
canvas = np.ones((512, 512, 3), dtype=np.uint8) * 255
cv2.rectangle(canvas, (120, 160), (390, 360), (0, 0, 0), 3)
sketch = render_pencil_sketch(canvas)
out_path = SKETCH_DIR / 'demo_box_sketch.png'
cv2.imwrite(str(out_path), sketch)

meta = {
    'sketch_path': str(out_path),
    'cad_spec': example_cad_spec(),
    'source': 'synthetic',
}
meta_path = SKETCH_DIR / 'demo_box_sketch.json'
meta_path.write_text(json.dumps(meta, indent=2))
print('Wrote', out_path)
print('Wrote', meta_path)